### 1. Scraping Campus Placement Data
This cell scrapes the campus hiring placement tables for multiple academic years directly from the Dayananda Sagar College of Engineering (DSCE) website using `requests` and `BeautifulSoup`,
* web scraped the data of each academic year available

In [3]:

import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://www.dsce.edu.in/placements/campus-hiring"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "lxml")

academic_years = ["2025-26", "2024-25", "2023-24", "2022-23"]

all_data = {}

for year in academic_years:

    section = soup.find(id=year)

    if section is None:
        print(f"{year}: Section not found")
        continue

    table = section.select_one("table")

    if table is None:
        print(f"{year}: Table not found")
        continue

    rows = []

    for tr in table.select("tr"):
        cols = [td.get_text(" ", strip=True) for td in tr.select("td")]

        if len(cols) == 3:
            rows.append(cols)

    # Remove first and last row
    rows = rows[1:-1]

    df = pd.DataFrame(
        rows,
        columns=["Sl_No", "Company", "Students_Placed"]
    )

    all_data[year] = df

    # Export individual CSV
    df.to_csv(f"DSCE_{year}.csv", index=False)


# Display each academic year table only once
for year, df in all_data.items():
    print(f"\nAcademic Year: {year}")
    print("Total Rows:", len(df))
    display(df)



Academic Year: 2025-26
Total Rows: 336


,Sl_No,Company,Students_Placed
0,1,Infosys,87
1,2,Capgemini,74
2,3,LTIMindtree,50
3,4,ACMEGRADE,41
4,5,Cognizant,29
...,...,...,...
331,332,Wipfli India LLP,1
332,333,YUKEN INDIA,1
333,334,ZETA,1
334,335,ZScaler,1



Academic Year: 2024-25
Total Rows: 309


,Sl_No,Company,Students_Placed
0,1,Capgemini,101
1,2,Accenture,93
2,3,Infosys,38
3,4,LTI MIntree,34
4,5,EY GDS,33
...,...,...,...
304,305,Wayfair,1
305,306,Wayspire,1
306,307,Winupskill,1
307,308,Wipfli India,1



Academic Year: 2023-24
Total Rows: 327


,Sl_No,Company,Students_Placed
0,1,InternzValley,67
1,2,EdgeVerve,59
2,3,Accenture,44
3,4,Capgemini,31
4,5,Capgemini Techpub,25
...,...,...,...
322,323,Zeta,1
323,324,Zifo,1
324,325,ZoomRx,1
325,326,Zydex,1



Academic Year: 2022-23
Total Rows: 326


,Sl_No,Company,Students_Placed
0,1,Capgemini,136
1,2,Cognizant,118
2,3,Mindtree,70
3,4,Accenture,46
4,5,TheMath,43
...,...,...,...
321,322,Zensar,1
322,323,Zeta,1
323,324,Zomato,1
324,325,Zscaler,1


### 2. Combining Academic Year Datasets and removing unnecessary rows
* combined all 4 different dataframes to a single one
* removed the first and row from each   table

In [4]:
# Combine all academic-year DataFrames after removing first and last rows
combined_df = pd.concat(
    [df.iloc[1:-1].assign(Academic_Year=year)
     for year, df in all_data.items()],
    ignore_index=True
)

display(combined_df)

,Sl_No,Company,Students_Placed,Academic_Year
0,2,Capgemini,74,2025-26
1,3,LTIMindtree,50,2025-26
2,4,ACMEGRADE,41,2025-26
3,5,Cognizant,29,2025-26
4,6,Commonwealth Bank of Australia,29,2025-26
...,...,...,...,...
1285,321,ZekeLabs Technologies,1,2022-23
1286,322,Zensar,1,2022-23
1287,323,Zeta,1,2022-23
1288,324,Zomato,1,2022-23


### 3. Creating a Pivot Table by Company
* labelled and raw data after combining the dataset

In [5]:
pivot_df = pd.pivot_table(
    combined_df,
    index="Company",
    columns="Academic_Year",
    values="Students_Placed",
    aggfunc="sum",
    fill_value=0
)
display(pivot_df)

Academic_Year,2022-23,2023-24,2024-25,2025-26
Company,,,,
3D Product,0,2,0,0
75f,3,0,0,0
7pi Innovations,0,0,0,7
A.P.Moller - Maersk,0,0,0,3
AB InBev,0,0,2,0
...,...,...,...,...
serent,0,0,0,1
service now,0,2,0,0
smartED,0,0,2,0


## 4. Cleaning and Casting Column Types
* changed object datatype to int32
* saved memory space
* runs faster

In [6]:
cols = ['2022-23', '2023-24', '2024-25', '2025-26']
for col in cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors='coerce').astype('int32')

pivot_df.info()
display(pivot_df.head())

<class 'pandas.core.frame.DataFrame'>
Index: 1059 entries, 3D Product to winprotech IT solutions
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   2022-23  1059 non-null   int32
 1   2023-24  1059 non-null   int32
 2   2024-25  1059 non-null   int32
 3   2025-26  1059 non-null   int32
dtypes: int32(4)
memory usage: 24.8+ KB


Academic_Year,2022-23,2023-24,2024-25,2025-26
Company,,,,
3D Product,0,2,0,0
75f,3,0,0,0
7pi Innovations,0,0,0,7
A.P.Moller - Maersk,0,0,0,3
AB InBev,0,0,2,0


### 5. Calculating Row and Column Totals
*  ADD COLUMN_TOTAL   
*  SORT ROWS
*  NOW ADD ROW_TOTAL

In [7]:
# Add total column
pivot_df["ROW_TOTAL"] = pivot_df.sum(axis=1)

# Sort by TOTAL in descending order
pivot_df = pivot_df.sort_values(
    by="ROW_TOTAL",
    ascending=False
)

# Add TOTAL row at the bottom
pivot_df.loc["COLUMN_TOTAL"] = pivot_df.sum(axis=0)

# Display
display(pivot_df)

Academic_Year,2022-23,2023-24,2024-25,2025-26,ROW_TOTAL
Company,,,,,
Accenture,46,44,93,7,190
Cognizant,118,8,27,29,182
Capgemini,0,31,0,74,105
TCS,18,7,0,51,76
Mindtree,70,0,0,0,70
...,...,...,...,...,...
conceptia,0,1,0,0,1
Zydex,0,1,0,0,1
ZoomRx,0,1,0,0,1


### 6. Exporting Final Results
* data available in csv and excel format

In [8]:
pivot_df.to_csv("DSCE_Placement_Pivot.csv")

In [9]:
pivot_df.to_excel("DSCE_Placement_Pivot.xlsx")